In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulare eficientă a circuitelor stabilizator cu primitivele Qiskit Aer

<details>
<summary><b>Versiunile pachetelor</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Îți recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
Această pagină arată cum să folosești primitivele Qiskit Aer pentru a simula eficient circuitele stabilizator, inclusiv pe cele supuse zgomotului Pauli.

Circuitele stabilizator, cunoscute și sub denumirea de circuite Clifford, reprezintă o clasă restricționată importantă de circuite cuantice care pot fi simulate eficient în mod clasic. Există mai multe moduri echivalente de a defini circuitele stabilizator. O definiție este că un circuit stabilizator este un circuit cuantic format exclusiv din următoarele Gate-uri:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Measurement](../api/qiskit/circuit#qiskit.circuit.Measure)

Rețineți că folosind Hadamard și S putem construi orice Gate de rotație Pauli ([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate) și [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)) cu un unghi din mulțimea ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ (cu excepția fazei globale), astfel încât putem include și aceste Gate-uri în definiție.

Circuitele stabilizator sunt importante pentru studiul corecției erorilor cuantice. Posibilitatea de a le simula clasic le face utile și pentru verificarea ieșirii calculatoarelor cuantice. De exemplu, să presupunem că vrei să execuți un circuit cuantic cu 100 de Qubiți pe un calculator cuantic. Cum știi că acesta se comportă corect? Un circuit cuantic cu 100 de Qubiți depășește capacitățile simulării clasice prin forță brută. Modificând circuitul tău astfel încât să devină un circuit stabilizator, poți rula circuite pe calculatorul cuantic cu o structură similară cu circuitul dorit, dar pe care le poți simula pe un calculator clasic. Verificând ieșirea calculatorului cuantic pe circuitele stabilizator, poți căpăta încredere că acesta se comportă corect și pe circuitele non-stabilizator. Consultă [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3) pentru un exemplu practic al acestei idei.

[Simulare exactă și cu zgomot cu primitivele Qiskit Aer](simulate-with-qiskit-aer) arată cum să folosești [Qiskit Aer](https://qiskit.org/ecosystem/aer/) pentru a efectua simulări exacte și cu zgomot ale circuitelor cuantice generice. Consideră circuitul exemplu folosit în acel articol, un circuit cu 8 Qubiți construit cu [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2):

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

Folosind Qiskit Aer, am putut simula acest circuit cu ușurință. Cu toate acestea, să presupunem că setăm numărul de Qubiți la 500:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

Deoarece costul simulării circuitelor cuantice crește exponențial cu numărul de Qubiți, un circuit atât de mare ar depăși în general capacitățile chiar și ale unui simulator performant precum Qiskit Aer. Simularea clasică a circuitelor cuantice generice devine neviabilă când numărul de Qubiți depășește aproximativ 50 până la 100. Totuși, rețineți că circuitul efficient_su2 este parametrizat prin unghiuri pe Gate-urile $R_y$ și $R_z$. Dacă toate aceste unghiuri sunt din mulțimea ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$, atunci circuitul este un circuit stabilizator și poate fi simulat eficient!

În celula de mai jos, rulăm circuitul cu primitiva Sampler susținută de simulatorul de circuite stabilizator, folosind parametri aleși aleatoriu astfel încât circuitul să fie garantat un circuit stabilizator.

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Simulatorul de circuite stabilizator suportă și simularea cu zgomot, dar numai pentru o clasă restricționată de modele de zgomot. Mai precis, orice zgomot cuantic trebuie caracterizat printr-un canal de [eroare Pauli](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error). [Eroarea de depolarizare](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html) se încadrează în această categorie, deci poate fi simulată. Canalele de zgomot clasic, precum [eroarea de citire](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html), pot fi de asemenea simulate.

Celula de cod de mai jos rulează aceeași simulare ca anterior, dar de data aceasta specificând un model de zgomot care adaugă o eroare de depolarizare de 2% fiecărui Gate CX, precum și o eroare de citire care inversează fiecare bit măsurat cu o probabilitate de 5%.

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Acum, să folosim primitiva Estimator susținută de simulatorul stabilizator pentru a calcula valoarea de așteptare a observabilului $ZZ \cdots Z$. Datorită structurii speciale a circuitelor stabilizator, rezultatul este foarte probabil să fie 0.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>